# 02 - Carga: Raw a Bronce
## Homogenizacion de Formato a Delta Tables

---

### Objetivo de este notebook

Este notebook implementa la transicion de la **capa Raw** a la **capa Bronce** del pipeline.
Se toman los archivos JSON crudos almacenados en el volumen de Unity Catalog y se convierten
a formato **Delta Table**, el formato columnar optimizado de Delta Lake.

### Que se hace en la capa Bronce?

En la capa Bronce se aplican **transformaciones minimas** de caracter tecnico (no de negocio):

| Operacion | Justificacion |
|-----------|---------------|
| Conversion JSON a Delta | Formato optimizado para consultas distribuidas |
| Renombramiento de columnas | Eliminar espacios y caracteres especiales incompatibles con Delta |
| Adicion de trazabilidad | Registrar cuando y de que archivo se cargo cada registro |

**No** se realizan transformaciones de negocio (filtros, enriquecimiento, etc.)
para preservar los datos lo mas fieles posible a la fuente original.

### Tablas generadas

- `pubchem_bronce.propiedades_compuestos` - Propiedades fisicoquimicas de los 15 medicamentos
- `pubchem_bronce.bioactividad` - Resultados de bioensayos (decenas de miles de registros)

---
## 1. Importacion de Librerias y Configuracion

Importamos las funciones de PySpark necesarias y configuramos las rutas
del volumen de origen y el esquema de destino.

In [ ]:
from pyspark.sql import functions as F  # Funciones de transformacion de Spark SQL
import glob as modulo_glob               # Para buscar archivos por patron en el sistema de archivos
import os                                # Para operaciones con rutas de archivos

In [ ]:
# Detectar catalogo activo de Unity Catalog
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]

# Ruta del volumen donde se almacenan los archivos JSON crudos
RUTA_RAW = f"/Volumes/{CATALOGO}/pubchem_raw/bioactividad"

# Esquema de destino para las tablas Delta de la capa Bronce
ESQUEMA_BRONCE = "pubchem_bronce"

# Mostrar configuracion
print(f"Catalogo: {CATALOGO}")
print(f"Ruta Raw: {RUTA_RAW}")
print(f"Esquema Bronce: {ESQUEMA_BRONCE}")

---
## 2. Cargar Archivo de Propiedades de Compuestos

Se busca el archivo de propiedades **mas reciente** en el volumen Raw
(usando el patron `propiedades_*.json`). Si existen multiples ingestas,
se toma la ultima en orden alfabetico, que corresponde a la mas reciente
gracias al formato de marca de tiempo en el nombre.

Se agregan dos columnas de **trazabilidad** (metadata de auditoria):
- `_bronce_fecha_carga`: Momento exacto de la carga
- `_bronce_archivo_origen`: Nombre del archivo fuente

Las columnas de propiedades conservan sus nombres originales de PubChem
(por ejemplo `CID`, `MolecularFormula`, `MolecularWeight`, etc.).

In [ ]:
# Buscar todos los archivos de propiedades en el volumen Raw
# sorted() ordena alfabeticamente; el ultimo es el mas reciente por el formato YYYYMMDD_HHMMSS
archivos_props = sorted(modulo_glob.glob(f"{RUTA_RAW}/propiedades_*.json"))

# Validar que existe al menos un archivo de propiedades
if not archivos_props:
    raise FileNotFoundError(
        "No se encontraron archivos de propiedades en el volumen Raw. "
        "Ejecuta primero el notebook 01_ingesta_raw."
    )

# Seleccionar el archivo mas reciente (ultimo de la lista ordenada)
archivo_props = archivos_props[-1]
print(f"Archivo de propiedades seleccionado: {os.path.basename(archivo_props)}")

# Leer el archivo JSON con Spark
# La opcion 'multiline=true' es necesaria porque el JSON tiene formato legible (pretty-printed)
df_props = spark.read.option("multiline", "true").json(archivo_props)

# Agregar columnas de trazabilidad para auditoria
# Estas columnas registran cuando y de donde se cargo cada registro
df_props = df_props \
    .withColumn("_bronce_fecha_carga", F.current_timestamp()) \
    .withColumn("_bronce_archivo_origen", F.lit(os.path.basename(archivo_props)))

# Guardar como Delta Table en el esquema Bronce
# mode='overwrite' reemplaza la tabla completa en cada ejecucion
# overwriteSchema=true permite cambiar el esquema si la fuente cambia
df_props.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{ESQUEMA_BRONCE}.propiedades_compuestos")

# Confirmar la creacion de la tabla
conteo_props = df_props.count()
print(f"Tabla creada: {ESQUEMA_BRONCE}.propiedades_compuestos ({conteo_props} registros)")

---
## 3. Cargar Archivo de Bioactividad

El archivo de bioactividad contiene decenas de miles de registros de ensayos biologicos.
Las columnas originales del API de PubChem tienen nombres con **espacios y caracteres especiales**
(por ejemplo `Activity Value [uM]`, `Panel Member ID`), que no son compatibles con Delta Lake.

Se realiza un **renombramiento tecnico** de columnas a formato snake_case en minusculas.
Este renombramiento es puramente tecnico (no de negocio) y es necesario para la compatibilidad
con el formato Delta.

### Mapeo de columnas

| Nombre original (PubChem API) | Nombre en Bronce |
|-------------------------------|------------------|
| AID | aid |
| Panel Member ID | panel_member_id |
| SID | sid |
| CID | cid |
| Activity Outcome | activity_outcome |
| Target Accession | target_accession |
| Target GeneID | target_gene_id |
| Activity Value [uM] | activity_value_um |
| Activity Name | activity_name |
| Assay Name | assay_name |
| Assay Type | assay_type |
| PubMed ID | pubmed_id |
| RNAi | rnai |

In [ ]:
# Buscar el archivo de bioactividad mas reciente en el volumen Raw
archivos_bio = sorted(modulo_glob.glob(f"{RUTA_RAW}/bioactividad_*.json"))

# Validar existencia del archivo
if not archivos_bio:
    raise FileNotFoundError(
        "No se encontraron archivos de bioactividad en el volumen Raw. "
        "Ejecuta primero el notebook 01_ingesta_raw."
    )

# Seleccionar el archivo mas reciente
archivo_bio = archivos_bio[-1]
print(f"Archivo de bioactividad seleccionado: {os.path.basename(archivo_bio)}")

# Leer el archivo JSON con Spark
df_bio = spark.read.option("multiline", "true").json(archivo_bio)

# Renombrar columnas para compatibilidad con Delta Lake
# Los nombres originales del API contienen espacios y corchetes que Delta no soporta
df_bio = df_bio \
    .withColumnRenamed("AID", "aid") \
    .withColumnRenamed("Panel Member ID", "panel_member_id") \
    .withColumnRenamed("SID", "sid") \
    .withColumnRenamed("CID", "cid") \
    .withColumnRenamed("Activity Outcome", "activity_outcome") \
    .withColumnRenamed("Target Accession", "target_accession") \
    .withColumnRenamed("Target GeneID", "target_gene_id") \
    .withColumnRenamed("Activity Value [uM]", "activity_value_um") \
    .withColumnRenamed("Activity Name", "activity_name") \
    .withColumnRenamed("Assay Name", "assay_name") \
    .withColumnRenamed("Assay Type", "assay_type") \
    .withColumnRenamed("PubMed ID", "pubmed_id") \
    .withColumnRenamed("RNAi", "rnai")

# Agregar columnas de trazabilidad para auditoria
df_bio = df_bio \
    .withColumn("_bronce_fecha_carga", F.current_timestamp()) \
    .withColumn("_bronce_archivo_origen", F.lit(os.path.basename(archivo_bio)))

# Guardar como Delta Table en el esquema Bronce
df_bio.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{ESQUEMA_BRONCE}.bioactividad")

# Confirmar la creacion de la tabla
conteo_bio = df_bio.count()
print(f"Tabla creada: {ESQUEMA_BRONCE}.bioactividad ({conteo_bio:,} registros)")
print(f"Columnas renombradas para compatibilidad con Delta Lake")

---
## 4. Verificacion de las Tablas Creadas

Verificamos que ambas tablas se crearon correctamente mostrando:
1. Una muestra de datos de cada tabla
2. El esquema completo (nombres y tipos de columnas)

Esto permite detectar rapidamente si algun campo tiene un tipo incorrecto
o si faltan columnas esperadas.

In [ ]:
# Mostrar una muestra de la tabla de propiedades de compuestos
print("--- Propiedades de compuestos (Bronce) ---")
spark.table(f"{ESQUEMA_BRONCE}.propiedades_compuestos").show(5, truncate=True)

# Mostrar una muestra de la tabla de bioactividad
print("--- Bioactividad (Bronce) - muestra ---")
spark.table(f"{ESQUEMA_BRONCE}.bioactividad").show(5, truncate=True)

In [ ]:
# Mostrar el esquema (estructura de columnas y tipos) de ambas tablas
# Esto es util para verificar que los tipos de datos inferidos por Spark son correctos
print("--- Esquema de propiedades ---")
spark.table(f"{ESQUEMA_BRONCE}.propiedades_compuestos").printSchema()

print("--- Esquema de bioactividad ---")
spark.table(f"{ESQUEMA_BRONCE}.bioactividad").printSchema()

print("\nCarga Raw a Bronce completada exitosamente.")